# A Randomized Trial Is Not a Universal Policy

Synthetic, reproducible companion for article #12. The trial and target are **disjoint** populations, the treatment is randomized only in the trial, and the ground-truth target ATE is computed by Gaussian-Hermite quadrature. No real-person observations are present.

In [ ]:
import tempfile
from pathlib import Path
import numpy as np
from analysis import Design, oracle_effects, sample_cohorts, estimate_all, run_study, effective_sample_size
np.set_printoptions(precision=5, suppress=True)

## Population truth and an observed trial

The target ATE is an expectation over $S=0$, not over the randomized participants $S=1$. The condition $Y(a)\perp S\mid X,Z$ holds only if all relevant transport modifiers are measured and conditioning makes the outcome mechanisms comparable.

In [ ]:
truth = oracle_effects()
cohorts = sample_cohorts(Design(), np.random.default_rng(20261030))
estimates, weight_diagnostics = estimate_all(cohorts)
print({key: round(value, 6) for key, value in truth.items()})
print({key: round(value, 6) for key, value in estimates.items()})
assert truth['target_ate'] > truth['trial_ate'] + 0.4
assert 0 < weight_diagnostics['rich_ess'] < Design().n_trial

## Why an unmeasured effect modifier cannot be solved by modeling harder

When a binary unobserved $U$ changes both treatment effects and selection, even an oracle model using all *observed* variables leaves an identification gap.

In [ ]:
hidden = oracle_effects(hidden_selection=2.0, hidden_effect=1.2)
print('Oracle bias from omitted modifier:', round(hidden['oracle_transport_bias'], 6))
assert abs(hidden['oracle_transport_bias']) > 0.1
assert abs(oracle_effects(hidden_selection=0, hidden_effect=1.2)['oracle_transport_bias']) < 1e-10

## Regenerate an independent quick study

The temporary directory avoids modifying the published evidence. The full analysis can be regenerated separately with `python analysis.py`.

In [ ]:
with tempfile.TemporaryDirectory(prefix='transport-notebook-') as temporary:
    root = Path(temporary)
    metrics = run_study(root, quick=True)
    figures = sorted((root / 'figures').glob('*_quick.png'))
    tables = sorted((root / 'results').glob('*_quick.csv'))
    assert len(figures) == 8 and len(tables) == 12
    assert all(path.stat().st_size > 40 for path in [*figures, *tables])
    print('Regenerated', len(figures), 'figures and', len(tables), 'tables')
    print('Target ATE:', round(metrics['target_ate'], 6))